In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
from datetime import datetime

In [ ]:
metrodf = pd.read_csv("POWER_Point_Hourly_20260501_20260531_020d28N_085d78E_LST.csv", skiprows=12, header=0)
nodesdf = pd.read_csv("nodes.csv")
edgesdf = pd.read_csv("edges.csv")

metrodf['WVP'] = (metrodf['RH2M']/100) * 6.105 * np.exp(17.27*metrodf['T2M']/(237.7+metrodf['T2M']))
metrodf['AT2M'] = metrodf['T2M'] + 0.33 * metrodf['WVP'] - 0.70 * metrodf['WS10M'] - 4

In [18]:
metrodf.head()

,YEAR,MO,DY,HR,ALLSKY_SFC_SW_DWN,T2M,RH2M,WS10M,WVP,AT2M
0,2026,5,1,0,0.0,26.30,94.16,5.32,32.117200,29.174676
1,2026,5,1,1,0.0,26.17,94.57,5.28,32.010879,29.037590
2,2026,5,1,2,0.0,26.00,94.64,4.72,31.714895,29.161915
3,2026,5,1,3,0.0,25.85,94.92,3.70,31.528142,29.664287
4,2026,5,1,4,0.0,25.72,95.39,3.08,31.441633,29.939739


In [40]:
nodesdf.head()

,osmid,y,x,street_count,geometry,name
0,8065246087,20.274953,85.777867,1,POINT (85.7778668 20.2749529),RHR
1,8065246096,20.274912,85.778567,3,POINT (85.7785672 20.2749124),NaN
2,8065209270,20.276272,85.778456,3,POINT (85.7784557 20.2762723),NaN
3,8065209273,20.276157,85.775806,1,POINT (85.7758059 20.2761566),NaN
4,2997273652,20.275541,85.778512,3,POINT (85.7785119 20.2755415),NaN


In [41]:
edgesdf.head()

,u,v,shade cover,water points,key,osmid,access,highway,oneway,reversed,length,geometry,junction
0,8065246087,8065246096,0.5,0.0,0,1569667948,private,service,False,False,75.158115,"LINESTRING (85.7778668 20.2749529, 85.777906 2...",NaN
1,8065246096,14299139685,0.7,1.0,0,230784863,private,service,False,False,116.225300,"LINESTRING (85.7785672 20.2749124, 85.7786588 ...",NaN
2,8065246096,2997273652,0.7,0.0,0,230784863,private,service,False,True,70.190233,"LINESTRING (85.7785672 20.2749124, 85.7785119 ...",NaN
3,8065246096,8065246087,0.5,0.0,0,1569667948,private,service,False,True,75.158115,"LINESTRING (85.7785672 20.2749124, 85.7782859 ...",NaN
4,8065209270,2392366807,0.7,0.0,0,230784863,private,service,False,False,67.069064,"LINESTRING (85.7784557 20.2762723, 85.7785005 ...",NaN


In [43]:
date = [5, 10, 14]
params = metrodf[(metrodf['MO']==date[0]) & (metrodf['DY']==date[1]) & (metrodf['HR']==date[2])]
params

,YEAR,MO,DY,HR,ALLSKY_SFC_SW_DWN,ALLSKY_SFC_LW_DWN,T2M,QV2M,RH2M
230,2026,5,10,14,0.0,1.49,28.35,16.08,66.35


In [ ]:
def getAHPWeights():
    matrix = np.array([
        [1.0,   2.0,   2.0,   1.5,   3.0],  # 1. Distance (Primary baseline anchor)
        [0.5,   1.0,   1.5,   0.5,   2.0],  # 2. Apparent Temp (Macro thermal stress)
        [0.5,   0.67,  1.0,   0.33,  2.0],  # 3. Solar Radiation (Overhead sun load)
        [0.67,  2.0,   3.0,   1.0,   3.0],  # 4. Shade % (Critical local mitigation)
        [0.33,  0.5,   0.5,   0.33,  1.0]   # Water Points (Secondary relief bonus)
    ])

    col_sums = matrix.sum(axis=0)
    norm_matrix = matrix/col_sums
    weights = norm_matrix.mean(axis=1)
    criteria = ['Distance', 'Apparent Temperature', 'Solar Radiation', 'Shade', 'Water Points']
    weight_dict = {}
    for crit, w in zip(criteria, weights):
        print(f"{crit}: {w:.4f}")
        weight_dict[crit] = w
        
    return weight_dict

In [20]:
def calculateCosts(edgesdf, metrodf, weights, timedate):
    df = edgesdf.copy()

    target_month = timedate.month
    target_date = timedate.day
    target_hour = timedate.hour
    
    weather_row = metrodf[
        (metrodf['month'] == target_month) & 
        (metrodf['date'] == target_date) & 
        (metrodf['hour'] == target_hour)
    ]
    
    if weather_row.empty:
        raise ValueError(f"No weather data found for Month: {target_month}, Date: {target_date}, Hour: {target_hour}")
    
    apparent_temp = weather_row['AT2M'].values[0]
    solar_radiation = weather_row['ALLSKY_SFC_SW_DWN'].values[0]

    norm_distance = df['length'] / df['length'].mean()

    temp_stress = max(0.0, (apparent_temp - 25.0) / 25.0)

    norm_solar = np.clip(solar_radiation / df['ALLSKY_SFC_SW_DWN'].max(), 0.0, 1.0)

    shade_ratio = np.clip(df.get('shade cover', 0.0) / 100.0, 0.0, 1.0)

    water_count = df.get('water points', 0.0)

    w_dist = weights.get('Distance', 0.3)
    w_temp = weights.get('Apparent Temperature', 0.2)
    w_solar = weights.get('Solar Radiation', 0.15)
    w_shade = weights.get('Shade %', 0.25)
    w_water = weights.get('Water Points', 0.1)

    environmental_load = 1.0 + (w_temp * temp_stress) + (w_solar * norm_solar)
    shade_mitigation = 1.0 - (w_shade * shade_ratio)
    shade_mitigation = np.maximum(shade_mitigation, 0.2)
    water_relief = w_water * water_count * 10.0

    base_scale = df['length']
    df['cost'] = (base_scale * norm_distance * environmental_load * shade_mitigation) - water_relief

    df['cost'] = np.maximum(df['cost'], 1.0)

    return df

    

In [44]:
temp_penalty = max(0, params['T2M'].item()-22) * 0.08
humidity_penalty = params['RH2M'] * 0.15

heat_score = 1 + temp_penalty + humidity_penalty
edgesdf['cost'] = edgesdf['length'] * heat_score.iloc[0]


In [ ]:
edgesdf.head()

In [ ]:
import networkx as nx

def dijkstra_route(edgesdf, snode, tnode):
    G = nx.from_pandas_edgelist(
        edgesdf,
        source='u',
        target='v',
        edge_attr=True
    )

    try:
        opt_path_nodes = nx.shortest_path(G, source=snode, target=tnode, weight='cost')
        opt_total_length = nx.path_weight(G, opt_path_nodes, weight='length')

        short_path_nodes = nx.shortest_path(G, source=snode, target=tnode, weight='length')
        short_total_length = nx.path_weight(G, short_path_nodes, weight='length')
        return (opt_path_nodes, opt_total_length), (short_path_nodes, short_total_length)

    except nx.NetworkXNoPath:
        print(f"Error: No valid path found between node {snode} and node {tnode}.")
        return [], float('inf')

    except nx.NodeNotFound as e:
        print(f"Error: Start or target node not found in graph: {e}")
        return [], float('inf')

In [24]:
def findNode(nodesdf, ustr):
    matches = nodesdf.loc[nodesdf['name'] == ustr, 'osmid']
    
    if not matches.empty:
        osmid = matches.values[0]
        return osmid if osmid else 0
        
    return 0

In [25]:
ustr = "Textiile dept."
vstr = "Gate 1"

u = findNode(nodesdf, ustr)
v = findNode(nodesdf, vstr)

print(u, v)

14299139697 2392366813


In [26]:
opt, short = dijkstra_route(edgesdf, u, v)

In [27]:
opt

([np.int64(14299139697),
  14299139689,
  2392366855,
  14299145808,
  2392366858,
  14263551058,
  14299139678,
  14299139674,
  2392366888,
  np.int64(2392366813)],
 1020.28658823)

In [28]:
short

([np.int64(14299139697),
  2392366876,
  10669422297,
  2392366840,
  8065209270,
  2392366807,
  2997273652,
  8065246096,
  14299139685,
  2392366900,
  np.int64(2392366813)],
 855.0551098899999)